Symetryczny system szyfrowania to taki, w którym klucz szyfrujący pozwala zarówno szyfrować dane, jak również odszyfrowywać je Podstawową wadą systemów symetrycznych jest ścisła konieczność ochrony klucza. Z tego powodu można je było stosować tylko w ograniczonych grupach użytkowników.

Algorytm RSA jest niesymetryczny, którego zasadniczą cechą są dwa klucze: publiczny do kodowania informacji oraz prywatny do jej odczytywania. Klucz publiczny umożliwia jedynie zaszyfrowanie danych i w żaden sposób nie ułatwia ich odczytania, nie musi być chroniony

Drugi klucz (prywatny, przechowywany pod nadzorem) służy do odczytywania informacji zakodowanych przy pomocy pierwszego klucza. Klucz ten nie jest udostępniany publicznie.

Algorytm RSA składa się z trzech podstawowych kroków:

Generacja klucza publicznego i tajnego. Klucz publiczny jest przekazywany wszystkim zainteresowanym i umożliwia zaszyfrowanie danych. Klucz tajny umożliwia rozszyfrowanie danych zakodowanych kluczem publicznym. Jest przechowywany on bezpiecznie bez możliwości udostępniania.

Użytkownik po otrzymaniu klucza publicznego koduje za jego pomocą swoje dane i przesyła je w postaci szyfru RSA do  adresata dysponującego kluczem tajnym

Klucz publiczny nie musi być chroniony, ponieważ nie umożliwia on rozszyfrowania informacji – proces szyfrowania nie jest odwracalny przy pomocy tego klucza.
Adresat po otrzymaniu zaszyfrowanej wiadomości rozszyfrowuje ją za pomocą klucza tajnego.




In [111]:
import struct
import zlib

from Crypto.Util.number import getPrime
import math

p i q to Liczby pierwsze
phi to funkcja Eulera, n jest modułem

In [112]:
def nwd(a,b):
    while b != 0:
        a,b = b, a % b
    return a
def find_e(phi_n):
    e = 3
    while nwd(e ,phi_n) != 1:
        e+= 2
    return e
def generate_rsa_keys(bits=1024):

    p = getPrime(bits // 2)
    q = getPrime(bits // 2)

    n = p * q
    phi_n = (p-1) * (q-1)

    #e = 65537
    e = find_e(phi_n)

    # rozszerzony algorytm Euklidesa jako funkcja pow() w Pythonie pozwala na obliczenie odwrotności modularnej. Operacaja potęgi modulo
    #Odwrotnośc modulo - liczba x, która po pomnożeniu przez zadaną liczbę "a" daje reśztę z dzielenia przez n równą 1
    d = pow(e, -1, phi_n)

    public_key = (e,n)
    private_key = (d,n)

    return public_key, private_key

In [113]:
def encrypt_rsa(message, public_key):
    e, n = public_key

    c = pow(message,e,n)
    return c

In [114]:
def decrypt_rsa(ciphertext, private_key):
    d, n = private_key

    m = pow(ciphertext,d,n)
    return m

In [115]:
def get_combined_idat(filepath):
    combined_idat_data = bytearray()

    with open(filepath, 'rb') as file:
        signature = file.read(8)

        while True:
            length_bytes = file.read(4)
            if not length_bytes:
                break

            chunk_length = struct.unpack(">I", length_bytes)[0]
            chunk_type = file.read(4).decode('ascii')
            chunk_data = file.read(chunk_length)
            chunk_crc = file.read(4)

            if chunk_type == 'IDAT':
                combined_idat_data.extend(chunk_data)
            elif chunk_type == 'IEND':
                break
        return bytes(combined_idat_data)

- input musi być zawsze mniejszy niż rozmiar klucza, stąd 100 bajtów dla 1024 bitowego klucza
- output musi być zawsze równy rozmiarowi klucza, stąd 128 bajtów dla 1024 bitowego klucza

In [116]:
def encrypt_bytes_ecb(data_bytes, pub_key, in_bloc_size=100, out_block_size=128):
    encrypted_data = bytearray()

    for i in range(0, len(data_bytes), in_bloc_size):
       block = data_bytes[i:i+in_bloc_size]

       m = int.from_bytes(block, byteorder='big')
       c = encrypt_rsa(m, pub_key)

       encrypted_data.extend(c.to_bytes(out_block_size,byteorder='big'))
    return bytes(encrypted_data)

In [117]:
import binascii
def create_png_chunk(chunk_type, chunk_data):
    type_bytes = chunk_type.encode('ascii')
    crc = binascii.crc32(type_bytes + chunk_data)

    length_bytes = struct.pack('>I', len(chunk_data))
    crc_bytes = struct.pack('>I', crc)

    return length_bytes + type_bytes + chunk_data + crc_bytes

In [118]:
import struct

def save_encrypted_png(input_filepath, output_filepath, new_idat_chunk):
    with open(input_filepath, 'rb') as infile, open(output_filepath, 'wb') as outfile:
        signature = infile.read(8)
        outfile.write(signature)

        idat_written = False

        while True:
            length_bytes = infile.read(4)
            if not length_bytes:
                break

            chunk_length = struct.unpack('>I', length_bytes)[0]
            chunk_type_bytes = infile.read(4)
            chunk_type = chunk_type_bytes.decode('ascii')
            chunk_data = infile.read(chunk_length)
            chunk_crc = infile.read(4)

            original_chunk = length_bytes + chunk_type_bytes + chunk_data + chunk_crc

            if chunk_type == 'IDAT':
                if not idat_written:
                    outfile.write(new_idat_chunk)
                    idat_written = True
            else:
                outfile.write(original_chunk)

            if chunk_type == 'IEND':
                break

In [119]:
def process_png_method_a(idat_compressed_bytes, pub_key):
    raw = zlib.decompress(idat_compressed_bytes)
    encrypted = encrypt_bytes_ecb(raw, pub_key)
    recompressed_data = zlib.compress(encrypted)

    return create_png_chunk('IDAT', recompressed_data)

In [120]:
def process_png_method_b(idat_compressed_bytes, pub_key):
    encrypted = encrypt_bytes_ecb(idat_compressed_bytes, pub_key)

    return create_png_chunk('IDAT', encrypted)

In [121]:
def decrypt_bytes_ecb(encrypted_data_bytes, priv_key, in_block_size=128, out_block_size=100):
    decrypted_data = bytearray()

    for i in range(0, len(encrypted_data_bytes), in_block_size):
       block = encrypted_data_bytes[i:i+in_block_size]

       c = int.from_bytes(block, byteorder='big')
       m = decrypt_rsa(c, priv_key)

       decrypted_data.extend(m.to_bytes(out_block_size,byteorder='big'))
    return bytes(decrypted_data)


In [122]:
def process_png_decrypt_method_a(encrypted_idat_bytes, priv_key):
   encrypted = zlib.decompress(encrypted_idat_bytes)
   raw = decrypt_bytes_ecb(encrypted, priv_key)
   recompressed_data = zlib.compress(raw)
   return create_png_chunk('IDAT',recompressed_data)

In [123]:
def process_png_decrypt_method_b(encrypted_idat_bytes, priv_key):
    decrypted_compressed_data = decrypt_bytes_ecb(encrypted_idat_bytes, priv_key)

    return create_png_chunk('IDAT', decrypted_compressed_data)

In [124]:
filepath = "photos/apple.png"
pub_key, priv_key = generate_rsa_keys(bits=1024)

original_combined_idat = get_combined_idat(filepath)

encrypted_idat_a = process_png_method_a(original_combined_idat, pub_key)
crypted_path_a = "cryptedapple_method_a.png"
save_encrypted_png(input_filepath=filepath, output_filepath=crypted_path_a, new_idat_chunk=encrypted_idat_a)

read_encrypted_idat_a = get_combined_idat(crypted_path_a)
decrypted_idat_a = process_png_decrypt_method_a(read_encrypted_idat_a, priv_key)
save_encrypted_png(crypted_path_a, "decrypted_apple_method_a.png", decrypted_idat_a)

encrypted_idat_b = process_png_method_b(original_combined_idat, pub_key)
crypted_path_b = "cryptedapple_method_b.png"
save_encrypted_png(input_filepath=filepath, output_filepath=crypted_path_b, new_idat_chunk=encrypted_idat_b)

read_encrypted_idat_b = get_combined_idat(crypted_path_b)
decrypted_idat_b = process_png_decrypt_method_b(read_encrypted_idat_b, priv_key)
save_encrypted_png(crypted_path_b, "decrypted_apple_method_b.png", decrypted_idat_b)